# RAG (Answer) Evaluation — LLM-as-Judge

Compares prompt strategies and two LLMs on a ~200-question sample using
a 3-way LLM-as-judge label scheme: `RELEVANT` / `PARTLY_RELEVANT` /
`NON_RELEVANT`. Requires `OPENAI_API_KEY`.

Report the label distribution as percentages per configuration and use
it to justify the final model choice (e.g. if gpt-4o-mini reaches ~90%+
RELEVANT, the cheaper model wins).

In [ ]:
import sys
sys.path.append("..")

import json
import random

import pandas as pd
from tqdm.auto import tqdm

from zoning_assistant import rag as ragmod

In [ ]:
gt = pd.read_csv("../data/ground-truth-retrieval.csv")
sample = gt.sample(n=200, random_state=1).to_dict(orient="records")
len(sample)

## Judge prompt (answer-only relevance, 3 labels)

In [ ]:
judge_template = """
You are an expert evaluator for a RAG system that answers municipal
zoning questions. Classify the relevance of the generated answer to the
question.

Question: {question}
Generated Answer: {answer}

Provide the output in parsable JSON without using code blocks:

{{"Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "[brief explanation]"}}
""".strip()


def judge(question, answer, model="gpt-4o-mini"):
    prompt = judge_template.format(question=question, answer=answer)
    raw, _, _ = ragmod.llm(prompt, model=model)
    return json.loads(raw)

## Evaluate a configuration end-to-end

In [ ]:
def evaluate_config(sample, answer_model):
    rows = []
    for q in tqdm(sample):
        out = ragmod.rag(q["question"], model=answer_model)
        verdict = judge(q["question"], out["answer"])
        rows.append({
            "question": q["question"],
            "answer": out["answer"],
            "model": answer_model,
            "relevance": verdict["Relevance"],
            "explanation": verdict["Explanation"],
            "cost": out["openai_cost"],
        })
    return pd.DataFrame(rows)

In [ ]:
df_mini = evaluate_config(sample, "gpt-4o-mini")
df_mini.relevance.value_counts(normalize=True)

In [ ]:
df_4o = evaluate_config(sample, "gpt-4o")
df_4o.relevance.value_counts(normalize=True)

## Sanity-check the judge

Hand-review a sample of judge labels to catch systematic bias, like the
judge rewarding longer answers regardless of correctness.

In [ ]:
df_mini.sample(10, random_state=7)[["question", "answer", "relevance", "explanation"]]

In [ ]:
df_mini.to_csv("../data/rag-eval-gpt-4o-mini.csv", index=False)
df_4o.to_csv("../data/rag-eval-gpt-4o.csv", index=False)